In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm
tqdm.pandas(leave = False)

from transformers import (
    AutoTokenizer,
    EsmTokenizer,
    EsmForMaskedLM,
    pipeline,
)

from itertools import chain
import torch
import torch.nn.functional as F
import scipy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

!pwd

In [ ]:
model_dict = {

    # replace with actual model paths
    # "uniform": "brineylab/uniform-250k",
    # "preferential": "brineylab/preferential-250k",

    "random": "../models/models/random-masking-model_2026-07-01"
}

# tokenizer
tokenizer = EsmTokenizer.from_pretrained("../tokenizer/vocab.txt")

In [ ]:
def infer_and_group_stats(model, tokenizer, seq, cdr):
    losses = []
    predictions = ""
    scores = []
    perplexities = []


    with torch.no_grad():
        sep = "<cls><cls>"
        sep_idx = seq.find(sep)
        heavy = seq[:sep_idx]
        light = seq[sep_idx + len(sep):]
        print("SEQ: ", seq) 
        print("CDR: ", cdr)
        
        cdr_mask = str(cdr)[:sep_idx] + str(cdr)[sep_idx + 2:]

        unmasked = tokenizer(seq, return_tensors = "pt").to(device)["input_ids"]
        ranges = [range(sep_idx), range(sep_idx + len(sep), len(seq))]
        total_len = sum(len(i) for i in ranges)

        # model iteratively predicts each residue (skipping over separator tokens)
        for i in chain(*ranges):
        # for i in tqdm(chain(*ranges), total=total_len, leave=False):
            masked = seq[:i] + "<mask>" + seq[i+1:]
            tokenized = tokenizer(masked, return_tensors="pt").to(device)
            mask_pos = (tokenized.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
            labels = torch.where(tokenized.input_ids == tokenizer.mask_token_id, unmasked, -100)
            output = model(**tokenized, labels = labels)
            logits = output.logits

            # predicted aa
            pred_token = logits[0, mask_pos].argmax(axis=-1)
            predictions+=tokenizer.decode(pred_token)
            print(tokenizer.decode(pred_token))
            # print("predicted token: ", pred_token)

            # prediction confidence
            prob = logits[0, mask_pos].softmax(dim=-1).topk(1)[0].item()
            scores.append(prob)

            # loss
            loss = output.loss.item()
            losses.append(loss)
            
            # perplexity
            ce_loss = F.cross_entropy(logits.view(-1, tokenizer.vocab_size), labels.view(-1)) # i think this is the same as output.loss.item()
            perplexities.append(float(torch.exp(ce_loss)))

        # group stats by region
        # find indices splitting regions (fwrs and cdrs in heavy and light chains)
        cdr_idxs = [0] + [i for i in range(len(cdr_mask)) if cdr_mask[i] != cdr_mask[i-1]] + [len(cdr_mask)]
        cdr_idxs.insert(7, sep_idx)
        
        # accuracy
        predictions_by_region = [predictions[cdr_idxs[n]:cdr_idxs[n+1]] for n in range(len(cdr_idxs)-1)]
        seq_by_region = [seq.replace(sep, "")[cdr_idxs[n]:cdr_idxs[n+1]] for n in range(len(cdr_idxs)-1)]
        region_mean_acc = [sum(true[i] == predict[i] for i in range(len(true)))/len(true) for true, predict in zip(seq_by_region, predictions_by_region)]

        # prediction confidence
        region_mean_scores = [np.mean(scores[cdr_idxs[n]:cdr_idxs[n+1]]) for n in range(len(cdr_idxs)-1)]

        # loss (median)
        region_median_loss = [np.median(losses[cdr_idxs[n]:cdr_idxs[n+1]]) for n in range(len(cdr_idxs)-1)]
        
        # perplexity (median)
        region_median_perplexity = [np.median(perplexities[cdr_idxs[n]:cdr_idxs[n+1]]) for n in range(len(cdr_idxs)-1)]
        
        return {
            "sequence": seq.replace(sep, ""),
            "cdr_mask": cdr_mask,
            "heavy": heavy,
            "light": light,
            "cdr_indices": cdr_idxs,
            "prediction": predictions,
            "accuracy_by_region": region_mean_acc,
            "score_by_region": region_mean_scores,
            "loss_by_region": region_median_loss,
            "perplexity_by_region": region_median_perplexity,
            "score": scores,
            "loss": losses,
            "perplexity": perplexities
        }

In [ ]:
for name, model_path in model_dict.items():
    for seq_type, data in data_dict.items():
    
        model = EsmForMaskedLM.from_pretrained(model_path).to(device)
    
        inference_data = []
        sequences = list(data.iterrows())

        for _id, row in tqdm(sequences):
            d = infer_and_group_stats(
                model, 
                tokenizer, 
                row['text'], 
                row['cdr_mask']
            )
            inference_data.append(d)
            
        inference_df = pd.DataFrame(inference_data)
        # saving to csv was creating typing problems (lists were strings) when loading in the data for further processing
                # inference_df.to_json(f"./results/{name}_{seq_type}_{len(inference_seqs)}.json")
        inference_df.to_json(f"./results/{name}_{seq_type}_{len(inference_data)}.json")

# Finds the recalculated BLOSUM scores for the inferences

### Creates a data frame for the inferences from all models used

In [ ]:
import pandas as pd
pref_germ_df = pd.read_json("./results/preferential_germline_1000.json", dtype=str)
pref_mut_df = pd.read_json("./results/preferential_mutated_1000.json", dtype=str)
uni_germ_df = pd.read_json("./results/uniform_germline_1000.json", dtype=str)
uni_mut_df = pd.read_json("./results/uniform_mutated_1000.json", dtype=str)

pref_germ_df["model"] = "Preferential"
pref_mut_df["model"] = "Preferential"
uni_germ_df["model"] = "Uniform"
uni_mut_df["model"] = "Uniform"

pref_germ_df["data_type"] = "Germline"
pref_mut_df["data_type"] = "Mutated"
uni_germ_df["data_type"] = "Germline"
uni_mut_df["data_type"] = "Mutated"


inference_df = pd.concat([pref_germ_df, uni_germ_df, pref_mut_df, uni_mut_df], join="inner")
inference_df

In [ ]:
import pandas as pd
import numpy as np
import condensed_blosum_metric
from condensed_blosum_metric import new_scores

# sequencelist = list(inference_df["sequence"])
inferredlist = list(inference_df["prediction"])
region = list(inference_df["cdr_mask"])
filelist = list(inference_df["model"])
file_type_list = list(inference_df["data_type"])

heavychainlist = list(inference_df["heavy"])
lightchainlist = list(inference_df["light"])

#sequencelist = heavychainlist
#sequencelist = lightchainlist

scores_list = []

for i in range (0, len(sequencelist)):
    find = new_scores(sequencelist[i], inferredlist[i], region[i])
    # find = new_scores(sequencelist[i], inferredlist[i][len(heavychainlist[i]):], region[i][len(heavychainlist[i]):])


    scores_list.append({
        "Seq ID": sequencelist[i],
        "Region": "Framework",
        "Metric": find[1],
        "Model": filelist[i],
        "Data Type": file_type_list[i],
        "Median": find[6]
    })

    scores_list.append({
        "Seq ID": sequencelist[i],
        "Region": "CDR1",
        "Metric": find[2],
        "Model": filelist[i],
        "Data Type": file_type_list[i],
        "Median": find[7]

    })

    scores_list.append({
        "Seq ID": sequencelist[i],
        "Region": "CDR2",
        "Metric": find[3],
        "Model": filelist[i],
        "Data Type": file_type_list[i],
        "Median": find[8]

    })

    scores_list.append({
        "Seq ID": sequencelist[i],
        "Region": "CDR3",
        "Metric": find[4],
        "Model": filelist[i],
        "Data Type": file_type_list[i],
        "Median": find[9]

    })



scores_df = pd.DataFrame(scores_list)

In [ ]:
scores_df